# Federated Reinforcement Learning (FedRL) — SAC cho Smart Grid

**Giai đoạn 2** trong lộ trình: `Centralized SAC → MARL IL → FedRL ← YOU ARE HERE`

## Tổng quan kiến trúc

```
╔══════════════════════════════════════════════════════╗
║              FEDERATED SERVER (Aggregator)           ║
║    w_global = FedAvg(w₁, w₂, w₃, w₄, w₅, w₆)       ║
╚═══════╦══════════════════════════════════╦═══════════╝
        ║ broadcast w_global               ║ receive w_i
        ▼                                  ▲
  ┌─────────┐  ┌─────────┐  ...  ┌─────────┐
  │  Node 1 │  │  Node 2 │       │  Node 6 │
  │ SAC_B1  │  │ SAC_B2  │       │ SAC_B6  │
  │ Env_B1  │  │ Env_B2  │       │ Env_B6  │
  │ data_B1 │  │ data_B2 │       │ data_B6 │
  └─────────┘  └─────────┘       └─────────┘
       ↕              ↕                  ↕
  Local train    Local train        Local train
  (K episodes)   (K episodes)       (K episodes)
```

## Nền tảng lý thuyết

### 1. Mục tiêu tối ưu hóa toàn cục

FedRL giải quyết bài toán:

$$\min_w F(w) = \sum_{i=1}^{N} \frac{n_i}{n} F_i(w)$$

Trong đó $F_i(w)$ là **objective function cục bộ** của agent $i$:

$$F_i(w) = -\mathbb{E}_{\tau \sim \pi_w^i} \left[ \sum_t r^i_t + \alpha \mathcal{H}(\pi_w^i(\cdot|o^i_t)) \right]$$

### 2. FedAvg Algorithm

Thuật toán **Federated Averaging (McMahan et al., 2017)** cho RL:

$$w^{r+1}_{global} = \sum_{i=1}^{N} \frac{n_i}{n} \cdot w^r_i$$

Sau đó broadcast: $w^{r+1}_i \leftarrow w^{r+1}_{global}$ (hoặc fine-tune thêm từ $w_{global}$)

| Ký hiệu | Ý nghĩa |
|---|---|
| $r$ | Round federation (1, 2, ..., R) |
| $w^r_i$ | Weights của agent $i$ sau round $r$ |
| $n_i$ | Số local steps agent $i$ thực hiện |
| $n = \sum n_i$ | Tổng local steps |
| $K$ | Số local episodes mỗi round |

### 3. Tại sao FedAvg giải quyết Non-stationarity?

Trong MARL IL, mỗi agent học trong môi trường "trôi nổi" vì các agent khác thay đổi. FedAvg đồng bộ hóa định kỳ:

$$\underbrace{w_i^r}_{\text{local policy}} \xrightarrow{\text{FedAvg}} \underbrace{w_{global}^r}_{\text{consensus}} \xrightarrow{\text{broadcast}} \underbrace{w_i^{r+1}}_{\text{new local start}}$$

Điều này tạo ra **implicit coordination** — mỗi agent "biết" về các agent khác qua shared weights, không cần share data thực tế.

### 4. Privacy Guarantee

Mỗi node chỉ chia sẻ **model weights** $w_i$, không chia sẻ:
- Raw data (điện tiêu thụ, nhiệt độ)
- Observations $o^i_t$
- Rewards $r^i_t$

Với **Differential Privacy** (phần mở rộng), thêm Gaussian noise trước khi upload:
$$\tilde{w}_i = w_i + \mathcal{N}(0, \sigma^2 \mathbf{I}), \quad \sigma = \frac{2 \cdot S \cdot \sqrt{2 \ln(1.25/\delta)}}{\epsilon}$$

### 5. FedProx (Cải tiến FedAvg)

**FedProx** (Li et al., 2020) thêm proximal term để ổn định khi data heterogeneous:

$$F_i^{prox}(w) = F_i(w) + \frac{\mu}{2} \|w - w_{global}\|^2$$

Giữ local weights gần với global → giảm client drift.

In [ ]:
import os
import glob
import copy
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch

# SB3
from stable_baselines3 import SAC
from stable_baselines3.common.vec_env import SubprocVecEnv
from stable_baselines3.common.monitor import Monitor

# CityLearn
from citylearn.citylearn import CityLearnEnv
from citylearn.wrappers import NormalizedObservationWrapper, StableBaselines3Wrapper

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()} (device: {'cuda' if torch.cuda.is_available() else 'cpu'})")

In [ ]:
# ================================================================
# FEDERATED RL — HYPERPARAMETERS
# ================================================================

# --- Federated settings ---
NUM_HOUSES              = 6       # Số node (tòa nhà)
NUM_ROUNDS              = 50      # Số vòng federation  R
LOCAL_EPISODES_PER_ROUND= 20      # K episodes mỗi round mỗi agent
EPISODE_LENGTH          = 168     # 1 tuần (giờ)
# → LOCAL_STEPS_PER_ROUND = K × T_ep = 20 × 168 = 3,360 steps/round/agent

# --- FedAvg settings ---
# Uniform weights: mỗi agent đóng góp như nhau
# Có thể đổi thành proportional nếu các nhà có lượng data khác nhau
AGGREGATION_WEIGHTS     = None    # None = uniform (1/N)

# --- Local SAC settings ---
NET_ARCH                = [256, 256]
BATCH_SIZE              = 256
BUFFER_SIZE             = 100_000
LEARNING_STARTS         = 500
NUM_CPU                 = 2       # Workers per agent

# --- Warm-start từ MARL IL checkpoints (Stage 1) ---
WARM_START_FROM_MARL    = True    # True: dùng weights từ marl_sac notebook

# --- Paths ---
BASE_SCHEMA             = "citylearn_challenge_2023_phase_3_1"
SESSION_NAME            = "fedrl_sac_fedavg"

MODEL_DIR               = Path.cwd() / "models" / SESSION_NAME
TENSORBOARD_DIR         = Path.cwd() / "training_logs"
MARL_MODEL_DIR          = Path.cwd() / "models" / "marl_sac_independent_learners"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Tính LOCAL_STEPS_PER_ROUND
LOCAL_STEPS_PER_ROUND   = (EPISODE_LENGTH * LOCAL_EPISODES_PER_ROUND * NUM_CPU)

print(f"{'='*62}")
print(f"  Federated RL — FedAvg")
print(f"{'='*62}")
print(f"  Rounds              : {NUM_ROUNDS}")
print(f"  Agents              : {NUM_HOUSES}")
print(f"  Local episodes/round: {LOCAL_EPISODES_PER_ROUND}")
print(f"  Local steps/round   : {LOCAL_STEPS_PER_ROUND:,}")
print(f"  Total steps (all)   : {NUM_ROUNDS * LOCAL_STEPS_PER_ROUND * NUM_HOUSES:,}")
print(f"  Warm-start MARL     : {WARM_START_FROM_MARL}")
print(f"{'='*62}")

## FedAvg Aggregator

Class `FedAvgAggregator` triển khai server-side logic:

### Weight Aggregation (chi tiết toán học)

SB3's SAC policy chứa các tensors:

```
policy.state_dict() keys:
  actor.latent_pi.0.weight  (256×obs_dim)   ← shared layers
  actor.latent_pi.0.bias    (256,)
  actor.latent_pi.2.weight  (256×256)
  actor.latent_pi.2.bias    (256,)
  actor.mu.weight           (act_dim×256)   ← mean head
  actor.mu.bias             (act_dim,)
  actor.log_std.weight      (act_dim×256)   ← std head  
  actor.log_std.bias        (act_dim,)
  critic.qf0.*              ...             ← Q1 network
  critic.qf1.*              ...             ← Q2 network
  critic_target.qf0.*       ...             ← target Q1
  critic_target.qf1.*       ...             ← target Q2
```

**FedAvg per tensor**:
$$w^{global}_k = \sum_{i=1}^{N} \alpha_i \cdot w^i_k, \quad \sum_{i=1}^N \alpha_i = 1$$

Với uniform aggregation: $\alpha_i = 1/N = 1/6 \approx 0.1\overline{6}$

**Ví dụ tính** với tensor `actor.latent_pi.0.bias` (256-dim):
```
Agent 1: b₁ = [0.12, -0.05, 0.33, ...]
Agent 2: b₂ = [0.08, -0.11, 0.28, ...]
...
Agent 6: b₆ = [0.15, -0.03, 0.31, ...]

b_global = (1/6) × (b₁ + b₂ + ... + b₆)
         = [0.11, -0.07, 0.30, ...]
```

In [ ]:
class FedAvgAggregator:
    """
    Server-side FedAvg aggregation cho SAC policies.

    Tách biệt hoàn toàn với môi trường và agent cục bộ —
    chỉ xử lý model weights (state_dict tensors).
    """

    def __init__(self, agent_ids: list, weights: list = None):
        """
        agent_ids : danh sách tên agent (building names)
        weights   : trọng số đóng góp Σαᵢ=1, None = uniform
        """
        self.agent_ids = agent_ids
        N = len(agent_ids)
        if weights is None:
            self.weights = {aid: 1.0 / N for aid in agent_ids}
        else:
            assert len(weights) == N and abs(sum(weights) - 1.0) < 1e-6
            self.weights = dict(zip(agent_ids, weights))
        self.global_state_dict = None
        self.round_history     = []   # log per-round metrics

    # ------------------------------------------------------------------
    def aggregate(self, local_state_dicts: dict) -> dict:
        """
        Thực hiện FedAvg:
            w_global[k] = Σᵢ αᵢ · wᵢ[k]   ∀ tensor key k

        local_state_dicts : {agent_id: OrderedDict từ policy.state_dict()}
        returns           : aggregated state_dict
        """
        reference = list(local_state_dicts.values())[0]
        global_sd = {}

        for key in reference.keys():
            # Kiểm tra tensor type — chỉ aggregate float tensors
            # (int tensors như batch norm counts không aggregate)
            stacked = torch.stack([
                local_state_dicts[aid][key].float() * self.weights[aid]
                for aid in self.agent_ids
            ])
            global_sd[key] = stacked.sum(dim=0).to(reference[key].dtype)

        self.global_state_dict = global_sd
        return global_sd

    # ------------------------------------------------------------------
    def broadcast(self, agents: dict):
        """
        Ghi global weights vào tất cả agents.
        agents : {agent_id: SAC model}
        """
        assert self.global_state_dict is not None, \
            "Cần gọi aggregate() trước khi broadcast()."
        for aid, agent in agents.items():
            agent.policy.load_state_dict(self.global_state_dict)

    # ------------------------------------------------------------------
    def compute_weight_divergence(self, local_state_dicts: dict) -> float:
        """
        Đo độ phân kỳ giữa các agent:
            Δ = (1/N) Σᵢ ||wᵢ - w_global||₂ / ||w_global||₂

        Giá trị cao → agents học rất khác nhau (data heterogeneous).
        Giá trị thấp → agents đã đồng thuận tốt.
        """
        if self.global_state_dict is None:
            return float("nan")
        total_div = 0.0
        for aid in self.agent_ids:
            diff_norm = sum(
                (local_state_dicts[aid][k].float() -
                 self.global_state_dict[k].float()).norm().item()
                for k in self.global_state_dict
            )
            glob_norm = sum(
                self.global_state_dict[k].float().norm().item()
                for k in self.global_state_dict
            )
            total_div += diff_norm / (glob_norm + 1e-8)
        return total_div / len(self.agent_ids)

    # ------------------------------------------------------------------
    def log_round(self, round_idx: int, metrics: dict):
        """Lưu metrics của mỗi round để vẽ đồ thị hội tụ."""
        entry = {"round": round_idx, **metrics}
        self.round_history.append(entry)

    def get_history_df(self) -> pd.DataFrame:
        return pd.DataFrame(self.round_history)

    # ------------------------------------------------------------------
    def save_global_weights(self, path: Path):
        """Lưu global weights (dùng cho resume hoặc deploy)."""
        torch.save(self.global_state_dict, path)
        print(f"  [Server] Global weights saved → {path.name}")

    def load_global_weights(self, path: Path):
        """Tải global weights từ file."""
        self.global_state_dict = torch.load(path, weights_only=True)
        print(f"  [Server] Global weights loaded ← {path.name}")


# Quick sanity check
print("FedAvgAggregator định nghĩa thành công.")
agg_test = FedAvgAggregator(["B1","B2","B3"], weights=[0.5, 0.3, 0.2])
print(f"  Test weights: {agg_test.weights}")

In [ ]:
# ================================================================
# SETUP: Environments & Agents
# ================================================================

def make_env(schema_dict, episode_time_steps):
    def _init():
        env = CityLearnEnv(
            schema=schema_dict, central_agent=True,
            episode_time_steps=episode_time_steps
        )
        env = NormalizedObservationWrapper(env)
        env = StableBaselines3Wrapper(env)
        env = Monitor(env)
        return env
    return _init

def get_single_building_schema(full_schema, building_name):
    schema = copy.deepcopy(full_schema)
    schema["buildings"] = {building_name: schema["buildings"][building_name]}
    return schema

# --- Tải schema ---
print("Đang tải schema...")
_tmp = CityLearnEnv(schema=BASE_SCHEMA)
FULL_SCHEMA    = copy.deepcopy(_tmp.schema)
BUILDING_NAMES = list(FULL_SCHEMA["buildings"].keys())[:NUM_HOUSES]
del _tmp

BUILDING_SCHEMAS = {
    n: get_single_building_schema(FULL_SCHEMA, n) for n in BUILDING_NAMES
}

# --- Probe dims ---
_probe = make_env(BUILDING_SCHEMAS[BUILDING_NAMES[0]], EPISODE_LENGTH)()
OBS_DIM = _probe.observation_space.shape[0]
ACT_DIM = _probe.action_space.shape[0]
_probe.close()

print(f"Buildings : {BUILDING_NAMES}")
print(f"Obs dim   : {OBS_DIM}  |  Act dim : {ACT_DIM}")

# --- Khởi tạo agents ---
agents   = {}
vec_envs = {}

for name in BUILDING_NAMES:
    node_dir = MODEL_DIR / name
    node_dir.mkdir(parents=True, exist_ok=True)

    env_fns = [make_env(BUILDING_SCHEMAS[name], EPISODE_LENGTH)
               for _ in range(NUM_CPU)]
    vec_env = SubprocVecEnv(env_fns)
    vec_envs[name] = vec_env

    # ── Warm-start từ MARL IL checkpoint ──────────────────────
    marl_path = MARL_MODEL_DIR / name / f"sac_{name}_final.zip"
    if WARM_START_FROM_MARL and marl_path.exists():
        agent = SAC.load(str(marl_path), env=vec_env,
                         tensorboard_log=str(TENSORBOARD_DIR))
        print(f"  [{name}] Warm-start từ MARL checkpoint ✓")

    # ── Resume từ FedRL checkpoint ────────────────────────────
    else:
        ckpts = [f for f in glob.glob(str(node_dir / "sac_*.zip"))
                 if "replay_buffer" not in f]
        if ckpts:
            latest = max(ckpts, key=os.path.getctime)
            agent = SAC.load(latest, env=vec_env,
                             tensorboard_log=str(TENSORBOARD_DIR))
            print(f"  [{name}] Resume từ FedRL checkpoint: {Path(latest).name}")
        else:
            agent = SAC(
                "MlpPolicy", vec_env,
                policy_kwargs=dict(net_arch=NET_ARCH),
                batch_size=BATCH_SIZE,
                buffer_size=BUFFER_SIZE,
                learning_starts=LEARNING_STARTS,
                verbose=0,
                tensorboard_log=str(TENSORBOARD_DIR),
            )
            print(f"  [{name}] Khởi tạo mới")

    agents[name] = agent

# --- Khởi tạo Aggregator ---
aggregator = FedAvgAggregator(BUILDING_NAMES, weights=AGGREGATION_WEIGHTS)
print(f"\nAggregation weights: {aggregator.weights}")
print(f"\nTất cả {len(agents)} agents sẵn sàng.")

## Federated Training Loop

### Thuật toán đầy đủ

```
FedRL-SAC (Algorithm):
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Input: R rounds, K episodes/round, N agents
Output: w_global (global policy)

1. Khởi tạo: w_global ← w_init  (hoặc MARL IL weights)

2. FOR round r = 1 to R:
   
   a. BROADCAST: wᵢ ← w_global  ∀i ∈ {1..N}
   
   b. LOCAL TRAIN (song song trong thực tế, tuần tự ở đây):
      FOR each agent i:
          wᵢ ← SAC.learn(K episodes, starting from wᵢ)
   
   c. COLLECT: {w₁, w₂, ..., wₙ}
   
   d. AGGREGATE:
          w_global ← Σᵢ αᵢ · wᵢ
   
   e. LOG: reward, divergence, KPI proxy
   
   f. SAVE checkpoint mỗi 5 rounds

3. RETURN w_global
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
```

### Lưu ý quan trọng về bước Broadcast

**Round 1**: Mỗi agent bắt đầu với MARL IL weights (warm-start).  
**Round 2+**: Mỗi agent **nhận lại** `w_global` từ server trước khi train.

Điều này khác biệt với MARL IL — ở đây agents "gặp nhau" mỗi $K$ episodes thông qua weight averaging, giải quyết non-stationarity.

In [ ]:
# ================================================================
# FEDERATED TRAINING LOOP
# ================================================================

# Kiểm tra resume: nếu đã có global weights thì tải lại
global_weights_path = MODEL_DIR / "global_weights_latest.pt"
history_path        = MODEL_DIR / "round_history.json"
start_round         = 0

if global_weights_path.exists():
    aggregator.load_global_weights(global_weights_path)
    aggregator.broadcast(agents)   # Broadcast weights đã lưu
    if history_path.exists():
        with open(history_path) as f:
            aggregator.round_history = json.load(f)
        start_round = len(aggregator.round_history)
    print(f"Resume từ round {start_round}/{NUM_ROUNDS}")
else:
    # Round 0: Thực hiện FedAvg lần đầu từ warm-start weights
    local_sds = {n: agents[n].policy.state_dict() for n in BUILDING_NAMES}
    aggregator.aggregate(local_sds)
    aggregator.broadcast(agents)
    print("Round 0: FedAvg khởi tạo từ warm-start weights.")

print(f"\nBắt đầu training từ round {start_round + 1} → {NUM_ROUNDS}")
print(f"{'─'*62}")

for round_idx in range(start_round + 1, NUM_ROUNDS + 1):
    print(f"\n[Round {round_idx:>3}/{NUM_ROUNDS}]", end="  ")

    # ── (a) BROADCAST w_global → agents (đã làm ở cuối round trước) ──
    # Lần đầu (round 1) thì đã broadcast từ round 0 ở trên

    # ── (b) LOCAL TRAIN ─────────────────────────────────────────────
    round_rewards = {}
    for name, agent in agents.items():
        agent.learn(
            total_timesteps=LOCAL_STEPS_PER_ROUND,
            reset_num_timesteps=False,   # Không reset bộ đếm global
            log_interval=None,           # Tắt log SB3 để output gọn
            tb_log_name=f"{SESSION_NAME}_{name}",
        )
        # Lấy mean reward gần nhất từ Monitor
        if hasattr(agent, "ep_info_buffer") and len(agent.ep_info_buffer) > 0:
            round_rewards[name] = np.mean(
                [ep["r"] for ep in agent.ep_info_buffer]
            )
        else:
            round_rewards[name] = float("nan")

    mean_reward = np.nanmean(list(round_rewards.values()))
    print(f"mean_ep_reward = {mean_reward:>8.2f}", end="  ")

    # ── (c) COLLECT local weights ────────────────────────────────────
    local_sds = {n: agents[n].policy.state_dict() for n in BUILDING_NAMES}

    # ── (d) AGGREGATE → w_global ────────────────────────────────────
    aggregator.aggregate(local_sds)

    # ── (e) Đo weight divergence (trước broadcast) ──────────────────
    divergence = aggregator.compute_weight_divergence(local_sds)
    print(f"divergence = {divergence:.4f}")

    # ── (f) BROADCAST w_global → agents ─────────────────────────────
    aggregator.broadcast(agents)

    # ── LOG ──────────────────────────────────────────────────────────
    aggregator.log_round(round_idx, {
        "mean_reward": mean_reward,
        "weight_divergence": divergence,
        **{f"reward_{n}": round_rewards.get(n, float("nan"))
           for n in BUILDING_NAMES},
    })

    # ── CHECKPOINT mỗi 5 rounds ──────────────────────────────────────
    if round_idx % 5 == 0 or round_idx == NUM_ROUNDS:
        aggregator.save_global_weights(global_weights_path)
        # Lưu per-agent checkpoint
        for name, agent in agents.items():
            ckpt_path = MODEL_DIR / name / f"sac_{name}_round{round_idx:03d}.zip"
            agent.save(str(ckpt_path))
        # Lưu history
        with open(history_path, "w") as f:
            json.dump(aggregator.round_history, f, indent=2)
        print(f"       → Checkpoint saved (round {round_idx})")

print(f"\n{'='*62}")
print("  FEDERATED TRAINING COMPLETE")
print(f"{'='*62}")

# Lưu final weights cho mỗi agent
for name, agent in agents.items():
    final_path = MODEL_DIR / name / f"sac_{name}_fedrl_final.zip"
    agent.save(str(final_path))
print("  Final models saved.")

## Convergence Analysis — Phân tích quá trình hội tụ

Theo dõi 2 chỉ số qua các round:

1. **Mean Episode Reward**: Tăng → policy đang cải thiện
2. **Weight Divergence** $\Delta$: Giảm → các agents đang đồng thuận

**Kỳ vọng lý thuyết**:
- Reward tăng dần và ổn định (không oscillate như MARL IL)
- Divergence giảm dần → bằng chứng FedAvg đang ép các agents về consensus

In [ ]:
hist_df = aggregator.get_history_df()

if hist_df.empty:
    print("Chưa có dữ liệu training. Hãy chạy cell Training trước.")
else:
    print("=== Convergence Summary ===")
    print(hist_df[["round", "mean_reward", "weight_divergence"]].to_string(index=False))

    # ── Tìm round hội tụ (reward cải thiện < 1%) ──────────────
    rewards = hist_df["mean_reward"].values
    converged_round = None
    for i in range(5, len(rewards)):
        window = rewards[i-5:i]
        improvement = (window[-1] - window[0]) / (abs(window[0]) + 1e-8)
        if abs(improvement) < 0.01:
            converged_round = int(hist_df.iloc[i]["round"])
            break

    print(f"\nRound hội tụ ước tính : {converged_round if converged_round else 'Chưa hội tụ'}")
    print(f"Best mean reward      : {rewards.max():.2f} (round {hist_df.loc[rewards.argmax(), 'round']})")
    print(f"Final divergence      : {hist_df['weight_divergence'].iloc[-1]:.4f}")

## Evaluation — Đánh giá FedRL trên joint environment

Quy trình đánh giá **giống hệt MARL IL**:
- Split joint obs 246-dim → 6 × 41-dim per agent
- Mỗi agent predict với `deterministic=True`
- Combine actions → 18-dim joint action

**Điểm khác biệt**: Agents FedRL đã được đồng bộ hóa qua `w_global` → kỳ vọng phối hợp tốt hơn MARL IL.

In [ ]:
# ================================================================
# EVALUATION
# ================================================================

# --- 1. Joint environment ---
joint_schema = copy.deepcopy(FULL_SCHEMA)
joint_schema["buildings"] = {n: joint_schema["buildings"][n]
                              for n in BUILDING_NAMES}

eval_env_raw  = CityLearnEnv(joint_schema, central_agent=True)
eval_env_norm = NormalizedObservationWrapper(eval_env_raw)
eval_env      = StableBaselines3Wrapper(eval_env_norm)

joint_obs_dim = eval_env.observation_space.shape[0]
joint_act_dim = eval_env.action_space.shape[0]
obs_per_agent = joint_obs_dim // NUM_HOUSES
act_per_agent = joint_act_dim // NUM_HOUSES

print(f"Joint obs : {joint_obs_dim}  |  obs/agent : {obs_per_agent}")
print(f"Joint act : {joint_act_dim}  |  act/agent : {act_per_agent}")

# --- 2. Tải final agents ---
print("\nĐang tải FedRL final agents...")
eval_agents = {}
for name in BUILDING_NAMES:
    p = MODEL_DIR / name / f"sac_{name}_fedrl_final.zip"
    if not p.exists():
        raise FileNotFoundError(f"Không tìm thấy: {p}\nHãy chạy Training trước.")
    eval_agents[name] = SAC.load(str(p))
    print(f"  ✓ {name}")

# --- 3. Rollout ---
obs, _ = eval_env.reset()
done = False
total_reward = 0.0
step_count   = 0

print("\nĐang chạy evaluation rollout...")
while not done:
    obs_per_building = [
        obs[i * obs_per_agent : (i+1) * obs_per_agent]
        for i in range(NUM_HOUSES)
    ]
    action_parts = []
    for i, name in enumerate(BUILDING_NAMES):
        a, _ = eval_agents[name].predict(
            obs_per_building[i].reshape(1, -1), deterministic=True
        )
        action_parts.append(a[0])

    joint_action = np.concatenate(action_parts).astype(np.float32)

    step_result = eval_env.step(joint_action)
    if len(step_result) == 5:
        obs, reward, terminated, truncated, _ = step_result
        done = terminated or truncated
    else:
        obs, reward, done, _ = step_result

    r = reward[0] if hasattr(reward, "__len__") else float(reward)
    total_reward += r
    step_count   += 1

print(f"Rollout hoàn tất — steps: {step_count:,}  |  total reward: {total_reward:.4f}")

In [ ]:
# ================================================================
# KPI EXTRACTION
# ================================================================
kpis = eval_env_raw.evaluate()
kpis_pivot = (
    kpis.pivot(index="cost_function", columns="name", values="value")
    .round(4).dropna(how="all")
)
print("=== FedRL — KPIs ===")
display(kpis_pivot)

In [ ]:
# ================================================================
# BẢNG SO SÁNH TOÀN DIỆN (4 phương pháp)
# ================================================================
KEY_METRICS = [
    "cost_total",
    "discomfort_proportion",
    "carbon_emissions_total",
    "electricity_consumption_total",
    "zero_net_energy",
    "all_time_peak_average",
    "ramping_average",
]

BASELINES = {
    "RBC (1 nhà)": {
        "cost_total": 2.013, "discomfort_proportion": 0.984,
        "carbon_emissions_total": 2.141, "electricity_consumption_total": 2.121,
        "zero_net_energy": 2.199, "all_time_peak_average": 1.138,
        "ramping_average": 1.091,
    },
    "SAC Centralized\n(1 nhà)": {
        "cost_total": 0.8959, "discomfort_proportion": 0.0657,
        "carbon_emissions_total": 0.9293, "electricity_consumption_total": 0.9282,
        "zero_net_energy": 0.8210, "all_time_peak_average": 1.0313,
        "ramping_average": 1.9068,
    },
    "SAC Centralized\n(6 nhà)": {
        "cost_total": 0.7947, "discomfort_proportion": 0.5415,
        "carbon_emissions_total": 0.8016, "electricity_consumption_total": 0.7991,
        "zero_net_energy": 0.7427, "all_time_peak_average": 0.9460,
        "ramping_average": 1.1309,
    },
    "MARL IL\n(6 nhà)": {
        m: float("nan") for m in KEY_METRICS  # Điền sau khi chạy marl_sac.ipynb
    },
}

# Lấy kết quả FedRL từ kpis_pivot
fedrl_district = {}
for m in KEY_METRICS:
    try:
        fedrl_district[m] = kpis_pivot.loc[m, "District"]
    except KeyError:
        fedrl_district[m] = float("nan")
BASELINES["FedRL FedAvg\n(6 nhà) ← HERE"] = fedrl_district

comparison_df = pd.DataFrame(BASELINES, index=KEY_METRICS)
comparison_df.index.name = "KPI"

# Tính % cải thiện FedRL so với Centralized 6-nhà
central_6 = comparison_df["SAC Centralized\n(6 nhà)"]
fedrl_col  = comparison_df["FedRL FedAvg\n(6 nhà) ← HERE"]
improvement = ((central_6 - fedrl_col) / central_6.abs() * 100).round(2)
comparison_df["Δ vs Central-6 (%)"] = improvement.apply(
    lambda x: f"+{x:.1f}%" if x > 0 else f"{x:.1f}%"
)

def highlight_best(s):
    numeric = s.apply(lambda x: float(str(x).replace("%","").replace("+",""))
                      if isinstance(x, str) and any(c.isdigit() for c in str(x))
                      else (x if isinstance(x, (int, float)) else float("nan")))
    is_best = numeric == numeric.min()
    return ["background-color: #c8f7c5; font-weight: bold" if v else ""
            for v in is_best]

display(
    comparison_df.style
    .apply(highlight_best, axis=1,
           subset=[c for c in comparison_df.columns if "Δ" not in c])
    .format({c: "{:.4f}" for c in comparison_df.columns if "Δ" not in c},
            na_rep="(run marl_sac)")
    .set_caption("So sánh tất cả phương pháp — District KPIs (thấp hơn = tốt hơn)")
)

## Kết luận & Hướng tiếp theo

### FedRL vs các phương pháp khác

| Góc độ | RBC | Central SAC | MARL IL | **FedRL** |
|---|---|---|---|---|
| Privacy | ✓ Local | ✗ Share obs | ✓ Local | ✓ Only weights |
| Scalability | ✓ | ✗ Retrain all | ✓ | ✓ Add node |
| Coordination | ✗ | ✓ Joint policy | ✗ | ✓ Via FedAvg |
| Non-stationarity | N/A | N/A | ✗ | ✓ Synced |
| Communication | None | Continuous | None | Periodic |
| Data heterogeneity | N/A | N/A | ✓ | ~ (FedProx cần) |

### Hyperparameters ảnh hưởng lớn nhất

- **`NUM_ROUNDS`**: Nhiều round → hội tụ tốt hơn nhưng tốn thời gian
- **`LOCAL_EPISODES_PER_ROUND` (K)**: K nhỏ → sync thường xuyên (ít drift), K lớn → train sâu hơn mỗi round (trade-off)
- **`AGGREGATION_WEIGHTS`**: Có thể dùng proportional weights dựa trên lượng data hoặc performance của từng nhà

### Công thức trade-off K

$$\text{Communication cost} \propto \frac{1}{K} \qquad \text{Client drift} \propto K$$

Optimal $K^*$ phụ thuộc vào mức độ heterogeneity của data giữa các buildings.

### Bước tiếp theo (Giai đoạn 3+)

1. **FedProx**: Thêm proximal regularization để xử lý data heterogeneous
2. **Personalized FedRL**: Mỗi nhà giữ layer cuối riêng, share các layer đầu
3. **Differential Privacy**: Clip + Gaussian noise trước khi upload weights
4. **Async Federated RL**: Không cần chờ tất cả agents — phù hợp thực tế